In [1]:
from pathlib import Path
from PIL import Image
import os

# 1) Paths (where your raw data is, and where processed data will go)
RAW_ROOT = Path("../data/raw/archive (1)/data1a")
OUT_ROOT = Path("../data/processed/archive1")

# 2) Target image size (standard for many CNN models)
TARGET_SIZE = (224, 224)

# 3) Classes (folder names in your dataset)
classes = ["00-damage", "01-whole"]

print("RAW_ROOT exists:", RAW_ROOT.exists())
print("RAW_ROOT:", RAW_ROOT.resolve())
print("OUT_ROOT:", OUT_ROOT.resolve())
print("TARGET_SIZE:", TARGET_SIZE)

# 4) Create output folders (if they don't exist)
for c in classes:
    (OUT_ROOT / "train" / c).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "val" / c).mkdir(parents=True, exist_ok=True)

# 5) Helper: count images in a folder (jpg/png/jpeg)
def count_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png"}
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in exts)

# 6) Check how many images you have in RAW (train/validation)
train_path = RAW_ROOT / "training"
val_path   = RAW_ROOT / "validation"

print("\n--- RAW COUNTS ---")
for c in classes:
    print(f"TRAIN {c}:", count_images(train_path / c))
for c in classes:
    print(f"VAL   {c}:", count_images(val_path / c))

print("\nIf the above counts look correct, next we will resize + save into data/processed/archive1/")


RAW_ROOT exists: True
RAW_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (1)\data1a
OUT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive1
TARGET_SIZE: (224, 224)

--- RAW COUNTS ---
TRAIN 00-damage: 920
TRAIN 01-whole: 920
VAL   00-damage: 230
VAL   01-whole: 230

If the above counts look correct, next we will resize + save into data/processed/archive1/


In [2]:
from pathlib import Path
from PIL import Image

RAW_ROOT = Path("../data/raw/archive (1)/data1a")
OUT_ROOT = Path("../data/processed/archive1")
TARGET_SIZE = (224, 224)
classes = ["00-damage", "01-whole"]

train_in = RAW_ROOT / "training"
val_in   = RAW_ROOT / "validation"

def process_split(split_in: Path, split_out: Path):
    exts = {".jpg", ".jpeg", ".png"}
    processed = 0
    skipped = 0

    for c in classes:
        in_dir  = split_in / c
        out_dir = split_out / c
        out_dir.mkdir(parents=True, exist_ok=True)

        for img_path in in_dir.rglob("*"):
            if img_path.suffix.lower() not in exts:
                continue

            try:
                img = Image.open(img_path).convert("RGB")          # consistent 3-channel image
                img = img.resize(TARGET_SIZE)                      # same size for all images

                out_path = out_dir / img_path.name                 # keep filename
                img.save(out_path, format="JPEG", quality=95)       # save as jpg

                processed += 1
            except Exception as e:
                skipped += 1

    return processed, skipped

print("Preprocessing TRAIN...")
train_processed, train_skipped = process_split(train_in, OUT_ROOT / "train")

print("Preprocessing VAL...")
val_processed, val_skipped = process_split(val_in, OUT_ROOT / "val")

print("\nDONE ✅")
print("Train processed:", train_processed, "| skipped:", train_skipped)
print("Val processed  :", val_processed,  "| skipped:", val_skipped)
print("\nSaved to:", OUT_ROOT.resolve())


Preprocessing TRAIN...
Preprocessing VAL...

DONE ✅
Train processed: 1840 | skipped: 0
Val processed  : 460 | skipped: 0

Saved to: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive1


In [ ]:
#checked the preprocessed data 
from pathlib import Path

train_damage = Path("../data/processed/archive1/train/00-damage")
train_whole  = Path("../data/processed/archive1/train/01-whole")
val_damage   = Path("../data/processed/archive1/val/00-damage")
val_whole    = Path("../data/processed/archive1/val/01-whole")

def count_images(path):
    return len(list(path.glob("*.jpg"))) + len(list(path.glob("*.JPEG")))

print("Processed TRAIN damage:", count_images(train_damage))
print("Processed TRAIN whole :", count_images(train_whole))
print("Processed VAL damage  :", count_images(val_damage))
print("Processed VAL whole   :", count_images(val_whole))


Processed TRAIN damage: 920
Processed TRAIN whole : 920
Processed VAL damage  : 230
Processed VAL whole   : 230
